In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from torchvision.transforms import Resize
from sklearn.cluster import KMeans

In [ ]:
def adaptive_mix(x_real, x_fake, alpha=1.0):
    lambda_val = torch.distributions.Beta(alpha, alpha).sample().to(x_real.device)
    mixed_x = lambda_val * x_real + (1 - lambda_val) * x_fake
    return mixed_x

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim, img_channels):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, img_channels * 32 * 32),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z).view(-1, 3, 32, 32)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_channels):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(img_channels * 32 * 32, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
def get_dataloader(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    dataset = torchvision.datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


def inception_score(images, device, num_splits=10):
    model = mobilenet_v2(weights="MobileNet_V2_Weights.IMAGENET1K_V1").to(device)
    model.classifier = nn.Identity()
    model.eval()

    resize = Resize((64, 64))  # Resize to fit MobileNetV2

    scores = []
    with torch.no_grad():
        for i in range(num_splits):
            batch = images[i::num_splits].to(device)
            batch = resize(batch)
            preds = model(batch)
            preds = F.softmax(preds, dim=1).cpu().numpy()
            kl_div = preds * (np.log(preds) - np.log(np.expand_dims(preds.mean(axis=0), 0)))
            scores.append(np.exp(kl_div.sum(axis=1).mean()))

    return np.mean(scores), np.std(scores)

In [ ]:
def train_gan(epochs=50, batch_size=128, alpha=1.0, z_dim=100, num_clusters=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataloader = get_dataloader(batch_size)
    img_channels = 3

    generator = Generator(z_dim, img_channels).to(device)
    discriminator = Discriminator(img_channels).to(device)

    criterion = nn.BCELoss()
    optimizer_g = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optimizer_d = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

    real_label, fake_label = 1, 0

    loss_g_list, loss_d_list, is_list = [], [], []

In [ ]:
 # Collect real images for clustering
    real_images_list = []
    for real_images, _ in dataloader:
        real_images_list.append(real_images.view(real_images.size(0), -1))
    real_images_data = torch.cat(real_images_list, dim=0).numpy()

    # Perform k-means clustering
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    kmeans.fit(real_images_data)
    cluster_labels = kmeans.labels_

    for epoch in range(epochs):
        for batch_idx, (real_images, _) in enumerate(dataloader):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

In [ ]:
# Assign cluster labels to real images
            real_flattened = real_images.view(batch_size, -1).cpu().numpy()
            assigned_clusters = kmeans.predict(real_flattened)

            z = torch.randn(batch_size, z_dim).to(device)
            fake_images = generator(z)

            mixed_images = adaptive_mix(real_images, fake_images, alpha)

            optimizer_d.zero_grad()
            real_output = discriminator(real_images)
            fake_output = discriminator(fake_images.detach())

            loss_d_real = criterion(real_output, torch.ones_like(real_output) * 0.9)
            loss_d_fake = criterion(fake_output, torch.zeros_like(fake_output))
            loss_d = (loss_d_real + loss_d_fake) / 2
            loss_d.backward()
            optimizer_d.step()

            optimizer_g.zero_grad()
            fake_output = discriminator(fake_images)
            loss_g = criterion(fake_output, torch.ones_like(fake_output) * 0.9)
            loss_g.backward()
            optimizer_g.step()

        is_mean, is_std = inception_score(fake_images, device)
        is_list.append(is_mean)
        loss_g_list.append(loss_g.item())
        loss_d_list.append(loss_d.item())
        print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {loss_d.item():.4f} | Loss G: {loss_g.item():.4f} | IS: {is_mean:.2f} ± {is_std:.2f}")

    plt.plot(is_list, label='Inception Score')
    plt.legend()
    plt.title('Inception Score Over Epochs')
    plt.show()

    with torch.no_grad():
        z = torch.randn(16, z_dim).to(device)
        gen_images = generator(z).cpu()

    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    for i, ax in enumerate(axes.flat):
        img = gen_images[i].permute(1, 2, 0).numpy()
        ax.imshow((img + 1) / 2)
        ax.axis('off')
    plt.show()


if __name__ == "__main__":
    train_gan(epochs=50)